# Сбор данных

Этот ноутбук - первый шаг: сбор отзывов клиентов о банках с сайта [banki.ru](https://www.banki.ru) для последующего сравнительного анализа банков.

**Что собираем:** отзывы по 5 банкам (Сбербанк, ВТБ, Альфа-Банк, Т-Банк, Совкомбанк) по всем продуктам для физических лиц за последние 12 месяцев.

**Зачем:** сайт banki.ru - открытый агрегатор отзывов с оценками и статусами решения проблем. Информация с этого сайта дает возможность сравнить банки между собой не только по формальному рейтингу, но и по содержанию отзывов клиентов.

**Результат ноутбука:** набор CSV-файлов (по одному на пару банк-продукт) с полями `id`, `title`, `userName`, `text`, `grade`, `commentCount`, `dateCreate`, `resolutionIsApproved`. Дальнейшая обработка - в ноутбуке **[2.Cleaning&structuring](2.Cleaning&structuring.ipynb)**.

In [1]:
import requests
import time
import json
import csv

#### Задача: спарсить отзывы с сайта banki.ru
Посмотрим, как выглядит страница с отзывами на конкретный банк и конкретную категорию этого банка (в дальнейшем будет интересны отзывы не по всем категориям)  
<img src="otziv1.png" width="400"> <img src="otziv2.png" width="400">

Нам интересны не все элементы, только:
* **Панель фильтров**: выпадающие списки для сортировки отзывов по услугам, типу проверки и оценке. Если ими поиграться, увидим, что фильтры меняют url. Если нужно будет парсить разные банки, категории, проверенные/не проверенные отзывы, с ответами от банка/или все -> просто смотрим, какие параметры передаются в запрос
* **Лента отзывов**: список самих отзывов пользователей с заголовками, оценками («1»), текстом жалобы, датой и статусами вроде «Проблема решена» или «Ответ банка». В ленте по дефолту 25 свежайших отзывов, сверху-вниз - от свежих до менее свежих

При нажати кнопки "Показать еще" подгружаются следующие по дате 25 отзывов. Осуществляется это через динамическую подгрузку через AJAX.  
Нажимаем кнопку "Показать еще" -> серверу улетает запрос ```https://www.banki.ru/services/responses/list/ajax/?lastId={id}&product={category}&is_countable=on&bank={bank}``` (параметры могут зависеть от фильтров)  
Ответ - JSON вида ```{"data": [...], "hasMorePages": bool, "lastId": int}```; lastId для след. запроса берётся из этого поля верхнего уровня (подробнее про ответ ниже)

### <span style="color: blue"> get_reviews_batch(bank, product, last_id=None, max_retries=3, timeout=15) </span>
#### функция забирает одну "порцию" отзывов с сайта banki.ru

* **bank** - банк, отзывы, которого парсим  
* **product** - продукт, отзывы по которому нас интересуют из списка ниже (список не исчерпывающий):  
    ['deposits', 'creditcards', 'hypothec', 'autocredits', 'credits']  
* **last_id** - с какого отзыва начало. Когда нужна "порция" самых новых отзывов, last_id = None. При повторных вызовах last_id уже будет числом - мы попросим следующую порцию  
* При сетевой ошибке (таймаут, обрыв соединения) повторяет попытку до **max_retries** раз с увеличивающейся паузой между попытками
* Если сервер не ответит за **timeout** секунд, это сразу считается ошибкой (и уходит в retry), а не зависает

#### Структура ответа от сервера
<pre>
{
  "data": [                          ← список отзывов
    {
      "id": int,                     ← уникальный id отзыва
      "title": str,                  ← заголовок отзыва
      "userName": str,               ← ник автора
      "text": str,                   ← текст отзыва
      "grade": int (1-5),            ← оценка  
      "commentCount": int,           ← число комментариев к отзыву
      "isCountable": bool,           ← учитывается ли в рейтинге банка
      "dateCreate": str,             ← дата - "ГГГГ-ММ-ДД ЧЧ:ММ:СС"
      "resolutionIsApproved": null,  ← разрешена ли проблема (не всегда заполнено)
      "agentAnswerText": str,        ← ответ банка на отзыв (если есть)
      "agentId": int,
      "hasDocuments": bool,
      "company": {                   ← вложенный объект с данными банка
        "id": int,
        "code": str,                 ← слаг банка, напр. "alfabank"
        "name": str,                 ← "Альфа-Банк"
        "url": str,
        "logo": str,
        "squareLogo": str,
        "phone": [ {...}, {...} ],   ← список телефонов банка (не нужно нам)
        "region": str,
        "address": str
      }
    },
    { ... },                         ← следующий отзыв, та же структура
    { ... }
  ],
  "hasMorePages": bool,              ← есть ли ещё страницы дальше
  "lastId": int                      ← id для запроса следующей порции
}
</pre>

In [2]:
def get_reviews_batch(bank, product, last_id=None, max_retries=3, timeout=15):
    url = "https://www.banki.ru/services/responses/list/ajax/"
    params = {"is_countable": "on", "bank": bank, "product": product}
    if last_id:
        params["lastId"] = last_id
 
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": f"https://www.banki.ru/services/responses/bank/{bank}/product/{product}/",
    }
 
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(
                url, params=params,
                headers=headers,
                timeout=timeout
            )
            resp.raise_for_status()
            print(f"    (статус ответа: {resp.status_code})")
            return resp.json()
        except (requests.exceptions.ConnectTimeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.ReadTimeout) as e:
            wait = attempt * 5  # 5, 10, 15 секунд
            print(f"  Ошибка сети (попытка {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                print(f"  Жду {wait} сек. и пробую снова...")
                time.sleep(wait)
            else:
                print("  Превышено число попыток - прокидываю ошибку дальше.")
                raise

Проверим функцию на одном запросе

In [3]:
batch = get_reviews_batch("alfabank", "deposits")

#batch['data']

#print(batch)
 
# Что вообще пришло
print("Ключи верхнего уровня ответа:", batch.keys())
print("hasMorePages:", batch.get("hasMorePages"))
print("lastId:", batch.get("lastId"))
print("Сколько отзывов в порции:", len(batch.get("data", [])))

    (статус ответа: 200)
Ключи верхнего уровня ответа: dict_keys(['data', 'hasMorePages', 'lastId'])
hasMorePages: True
lastId: 13285853
Сколько отзывов в порции: 25


**batch** - словарь c ключами: 'data', 'hasMorePages', 'lastId'  
**batch['data']** - список отзывов  
**batch['hasMorePages']** - есть ли еще отзывы дальше (bool)  
**batch['lastId']** - id последнего отзыва в ```batch['data']```

### <span style="color: blue"> save_to_csv(reviews, filepath) </span>
#### Сохраняет ключевые поля отзывов в CSV  
**reviews** - список отзывов  
**filepath** - путь, куда сохранить файл

In [4]:
def save_to_csv(reviews, filepath):
    fields = ["id", "title", "userName", "text", "grade", "commentCount", "dateCreate", "resolutionIsApproved"]
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        for r in reviews:
            writer.writerow(r)
    print(f"Сохранено {len(reviews)} отзывов в {filepath}")

### <span style="color: blue"> collect_all(bank, product, period_start, period_end=None, checkpoint_path=None, checkpoint_every=10) </span>
Собирает все отзывы для банка/продукта, начиная с самых свежих, пока не дойдёт до даты **period_start** (строка "ГГГГ-ММ-ДД")  
**period_end** - опционально, чтобы отсеять слишком свежие отзывы  
Если указан **checkpoint_path** - каждые **checkpoint_every** страниц сохраняет промежуточный результат в CSV, чтобы не потерять прогресс при сбое

In [30]:
def collect_all(bank, product, period_start, period_end=None, checkpoint_path=None, checkpoint_every=10):
    all_reviews = []
    last_id = None
    page = 0
 
    while True:
        page += 1
        print(f"Запрос страницы {page}, lastId={last_id}...")
        batch = get_reviews_batch(bank, product, last_id)
        reviews = batch.get("data", [])
 
        if not reviews:
            print("Пустой список отзывов. Диагностика - вот что реально пришло от сервера:")
            print(json.dumps(batch, ensure_ascii=False, indent=2)[:1000])
            print("Останавливаемся.")
            break
 
        for r in reviews:
            review_date = r["dateCreate"]
            if review_date >= period_start and (period_end is None or review_date <= period_end):
                all_reviews.append(r)
 
        last_date = reviews[-1]["dateCreate"]
        print(f"  получено {len(reviews)} отзывов, последняя дата в порции: {last_date}")
 
        # промежуточное сохранение
        if checkpoint_path and page % checkpoint_every == 0:
            save_to_csv(all_reviews, checkpoint_path)
            print(f"  [checkpoint] промежуточно сохранено {len(all_reviews)} отзывов")
 
        if last_date < period_start or not batch.get("hasMorePages", False):
            print("Достигли начала периода или страниц больше нет - останавливаемся.")
            break
 
        last_id = batch["lastId"]
        time.sleep(2.5)
 
    # финальное сохранение в любом случае
    if checkpoint_path:
        save_to_csv(all_reviews, checkpoint_path)
 
    return all_reviews

#### Собираем отзывы
По банкам из **BANKS** и по продуктам из **PRODUCTS** в период с **PERIOD_START** по **PERIOD_END**  
Результат сохраняется в отдельный csv-файл по каждой паре (банк, продукт) с именем **"bank_product.csv"**

In [ ]:
PERIOD_START = "2025-08-01"
PERIOD_END = None 

BANKS = ["sovcombank", "alfabank", "tcs", "vtb", "sberbank"] # список интересующих банков
# список интересующих продуктов
PRODUCTS = ["hypothec", "autocredits", "credits", "restructing", "deposits", "transfers", "currency_exchange", "remote", "other", "mobile_app", "individual", "creditcards", "debitcards"]

for bank in BANKS:
    for product in PRODUCTS:
        print("==========================================")
        print(f"{bank}: {product}")
        
        reviews = collect_all(bank, product, PERIOD_START, PERIOD_END, checkpoint_path=f"{bank}_{product}.csv", checkpoint_every=10)
 
        print(f"\nВсего собрано отзывов за период: {len(reviews)}")
 
        save_to_csv(reviews, f"{bank}_{product}.csv")

        print(f"Готово, сохранено в {bank}_{product}.csv")